### What are the types of joins and joins strategies in spark?

In Spark (PySpark / Spark SQL), interviewers typically ask this expecting an answer across two layers: **Logical Join Types** (the SQL syntax/semantic intent) and **Physical Join Strategies** (the execution algorithms under the Catalyst optimizer).

---

#### 1. Logical Join Types (Semantic Joins)

* **`inner` (Default):** Returns rows with matching keys in both DataFrames.
* **`left` / `left_outer`:** Returns all rows from the left DataFrame and matching values from the right (or `NULL` if no match).
* **`right` / `right_outer`:** Returns all rows from the right DataFrame and matching values from the left (or `NULL` if no match).
* **`full` / `full_outer`:** Returns all rows when there is a match in either the left or right side, filling mismatches with `NULL`.
* **`left_semi` / `semi`:** Returns only the columns and rows from the left DataFrame where a matching key exists in the right (acts like an `EXISTS` subquery; does not duplicate left rows on 1-to-many matches).
* **`left_anti` / `anti`:** Returns only the columns and rows from the left DataFrame that have **no** matching key in the right (acts like a `NOT EXISTS` subquery).
* **`cross`:** Computes the full Cartesian product ($M \times N$ rows). In newer versions, non-equi joins without explicit conditions trigger this or require `spark.sql.crossJoin.enabled=true`.

---

#### 2. Physical Join Strategies (Engine Execution Under the Hood)

Spark Catalyst selects from **5 physical join strategies** ranked by cost and dataset size:

| Strategy | When Spark Uses It | Key Characteristic / Advantage |
| --- | --- | --- |
| **Broadcast Hash Join (BHJ)** | One side is smaller than `spark.sql.autoBroadcastJoinThreshold` (default 10 MB) or forced via `broadcast()`. | **No shuffle.** Driver broadcasts the small table to all executor memory. Fastest join. |
| **Shuffle Hash Join (SHJ)** | One side is relatively smaller than the other (builds hash table per partition) and `preferSortMergeJoin` is false or AQE dynamically demotes SMJ. | Shuffles by key, builds in-memory hash table on small side per partition. Avoids sorting overhead. |
| **Shuffle Sort Merge Join (SMJ)** | Default for two large DataFrames joining on equi-join keys (`=`). | Shuffles both tables on join keys, **sorts** partitions, then merges iteratively. Highly robust against Out-Of-Memory (OOM) errors. |
| **Broadcast Nested Loop Join (BNLJ)** | Used for non-equi joins (`<`, `>`, `!=`) or when no join key is provided and one side can be broadcast. | Broadcasts one side and loops over every row of the other. Can be slow if datasets grow. |
| **Cartesian Product Join (CPJ)** | Full shuffle nested loop used for large-scale cross-joins without equi-conditions. | Shuffles all partitions across all executors. Heaviest compute cost. |

---

#### Key Takeaway for the Interview

- If asked about performance tuning: mention that **Adaptive Query Execution (AQE)** at runtime dynamically converts **Sort Merge Join (SMJ) into Broadcast Hash Join (BHJ)** if post-filter statistics show a table dropped below the broadcast threshold.

### How do you handle severe data skew during joins in PySpark and Databricks?

Severe data skew occurs when a specific join key is overwhelmingly frequent (e.g., `null`, default, or high-volume category IDs), causing most records to route to a single partition. This creates the classic **"99% completed task hangs on the last task"** executor bottleneck or triggers an Out-Of-Memory (OOM) error.

Here is how to tackle it in an interview setting, structured from **automatic runtime features** to **manual code-level techniques**.

---

#### 1. Enable Adaptive Query Execution (AQE) Skew Join Handling

In Databricks and modern Spark (3.0+), AQE is enabled by default. It detects skew partitions at runtime post-shuffle and automatically splits the skewed partition into smaller sub-partitions.

**Configuration:**
*   ```python
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    # Default skew thresholds (tweak if Spark fails to flag the partition as skewed):
    spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 5) # 5x median partition size
    spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "64MB")

    ```
---

#### 2. Manual Salting (The Gold-Standard Coding Answer)

If AQE cannot resolve the skew or you need programmatic control, use **salting** to distribute the hot key across multiple partitions.

##### Step-by-Step Logic:

1. On the **skewed (large) DataFrame**, append a random integer column (`salt`) between `0` and `N - 1` (e.g., $N = 5$).
2. On the **lookup/dimension DataFrame**, explode the rows using `array([lit(i) for i in range(N)])` so every lookup row duplicates with salt values `0` through `N - 1`.
3. Perform the join on `['join_key', 'salt']`.
4. Drop the `salt` column afterward.

**Code**

*   ```python
    from pyspark.sql import functions as F

    salt_factor = 5

    # 1. Add random salt to the skewed table
    df_large_salted = df_large.withColumn("salt", (F.rand() * salt_factor).cast("int"))

    # 2. Explode the smaller/lookup table to match all possible salt keys
    df_small_salted = df_small.withColumn("salt_array", F.array([F.lit(i) for i in range(salt_factor)])) \
                            .withColumn("salt", F.explode("salt_array")) \
                            .drop("salt_array")

    # 3. Join on both the original key and the salt
    df_joined = df_large_salted.join(df_small_salted, on=["id", "salt"], how="inner").drop("salt")

    ```
---

#### 3. Broadcast Hash Join (BHJ)

If one of the datasets is small enough to fit in executor memory:

* Eliminate the shuffle entirely by broadcasting the small table:
* ```python
    from pyspark.sql.functions import broadcast
    df_joined = df_large.join(broadcast(df_small), on="id", how="inner")

  ```
* Because all executors receive a full copy of the small table, no shuffle step happens, completely neutralizing key skew.

---

#### 4. Split-and-Union Pattern (Isolating Hot Keys)

If only a handful of specific keys (like `NULL` or `'UNKNOWN'`) cause 90% of the skew:

1. Filter the dataset into two subsets: `df_skewed` (only hot keys) and `df_unskewed`.
2. Handle the `df_unskewed` with a standard Sort-Merge / Shuffle Hash join.
3. Handle `df_skewed` separately (e.g., broadcast join, salting, or null-handling logic).
4. `unionByName()` both results.

---

#### 5. Databricks-Specific Optimizations (Delta Lake)

* **Z-Ordering / Liquid Clustering:** If queries frequently join or filter on the skewed key, apply Delta Liquid Clustering (`CLUSTER BY (join_key)`) or Z-Order to minimize data scanning and improve partition skipping.
* **Join Hints:** Use explicit Databricks skew hints if you know the exact skewed key:
    *   ```sql
        SELECT /*+ SKEW('df_large', 'id', ('NULL', 101, 102)) */ *
        FROM df_large
        JOIN df_small ON df_large.id = df_small.id

        ```

### What is this code doing? Is it Streaming, Batch, Incremental Load, or SCD?

![image_1788441771649.png](./image_1788441771649.png "image_1788441771649.png")

This code executes an **Incremental Batch Load** written using **Spark Structured Streaming syntax**.

It is **not** a continuous real-time stream, and it is **not** an SCD merge.

---

### Line-by-Line Breakdown of What It Does

```python
streaming_query = (parsed_streaming_df.writeStream.format("delta")
    .outputMode("Append")
    .option("checkpointLocation", "/Volumes/finguard/source/transactions/checkpoint/")
    .trigger(availableNow=True)
    .toTable("finguard.bronze.transactions_streaming_test")
)

```

1. **`writeStream.format("delta")`**: Uses the Spark Structured Streaming engine to write data directly into a **Delta Lake table**.
2. **`.outputMode("Append")`**: Only appends new records to the target table. It does not update existing rows or delete anything (which is why this is a raw Bronze layer ingest, not an SCD operation).
3. **`.option("checkpointLocation", "...")`**: Stores the state/offset bookmark in a Unity Catalog Volume. This directory records exactly which files or records have already been processed so they are never processed twice (guaranteeing **exactly-once processing**).
4. **`.trigger(availableNow=True)` (The Key Line)**:
    * This turns the streaming engine into an **Incremental Batch** job.
    * Instead of running forever waiting for data, it checks the source, processes **all new/unprocessed data currently available**, commits the checkpoint, and **stops completely**.


5. **`.toTable("finguard.bronze.transactions_streaming_test")`**: Targets a Bronze Delta table using Unity Catalog's 3-level namespace (`catalog.schema.table`).

---

### Why It Fits Each Concept

* **Is it Streaming?** It uses the *Structured Streaming API* (`writeStream`), but because of `availableNow=True`, it does **not run continuously**.
* **Is it Batch?** Yes, it behaves like a **batch job** that executes once and shuts down.
* **Is it Incremental Load?** **Yes.** Thanks to the `checkpointLocation`, every time this cell is executed, it only loads data that arrived since the last run.
* **Is it SCD (Slowly Changing Dimensions)?** **No.** It uses `Append` mode for a raw Bronze table. SCD requires a `MERGE INTO` statement with `WHEN MATCHED UPDATE` and `WHEN NOT MATCHED INSERT` logic (typically in Silver/Gold layers).

---

### How to Explain This Exact Code in an Interview

- This snippet performs an incremental Bronze ingestion into a Delta Lake table using Spark Structured Streaming. By specifying `trigger(availableNow=True)`, we achieve cost-efficient incremental batch processing: the job identifies all new records since the last checkpoint, appends them to the target table, updates the checkpoint state, and cleanly terminates without keeping a streaming cluster running 24/7.

### How do I untangle "Streaming," "Incremental," "Auto Loader," and "SCD"?

The confusion comes from the fact that modern data platforms use the same API for two different operational models.

A simple mental framework untangles the naming overlap:

**The Engine vs. The Schedule**

* **Structured Streaming (`readStream` / `writeStream`)** is the **engine**. It provides offset tracking, state management, and checkpointing so you never re-read old records.
* **The Trigger** is the **schedule**:
    * `trigger(processingTime='10 seconds')` -> **Continuous Streaming** (24/7 cluster, sub-minute latency).
    * `trigger(availableNow=True)` -> **Incremental Batch** (spin up, process new records, checkpoint, shut down).



**Where the Terms Actually Fit**

| Term | What It Actually Means | Where It Lives |
| --- | --- | --- |
| **Streaming** | Ingesting and moving data record-by-record or micro-batch with low latency. | Ingestion / Processing (`writeStream`) |
| **Incremental Load** | Processing only delta records (new or changed) instead of the entire historical dataset. | Architectural Goal (achieved via Checkpoints, CDF, or Watermarks) |
| **Auto Loader** | Databricks' file discovery tool (`cloudFiles`) that detects new storage files without directory scans. | Source Ingestion Layer (`spark.readStream.format("cloudFiles")`) |
| **SCD (Type 1 / Type 2)** | Managing historical updates to dimensions (overwriting vs. adding version rows with valid date ranges). | Transformation / Silver-Gold Layer (`MERGE INTO`) |

Keeping the operational engine separate from the data modeling concept (like SCD) prevents the terminology from blurring together.

### Auto Loader vs. Structured Streaming

#### 1. Core Relationship & Hierarchy

* **Structured Streaming** is the **core streaming framework** provided by Apache Spark.
* **Auto Loader (`cloudFiles`)** is a **specialized file source connector** developed by Databricks that plugs directly into Spark Structured Streaming.
* **Relationship:** Auto Loader does not replace Structured Streaming; it runs **on top of** Structured Streaming specifically to solve the problem of ingesting cloud storage files.

```
+-------------------------------------------------------------------------------+
|                    Apache Spark Structured Streaming (The Engine)             |
|                                                                               |
|   Supported Sources:                                                          |
|   ├── Kafka (`format("kafka")`)                                               |
|   ├── Delta Lake (`format("delta")`)                                          |
|   ├── Standard Spark File Source (`format("parquet")`, `format("json")`)      |
|   └── Databricks Auto Loader (`format("cloudFiles")`) <── [Proprietary Plug]   |
+-------------------------------------------------------------------------------+

```

---

#### 2. Key Differences Comparison

| Feature | Standard Structured Streaming (Files) | Databricks Auto Loader (`cloudFiles`) |
| --- | --- | --- |
| **Code Syntax** | `spark.readStream.format("json").load(path)` | `spark.readStream.format("cloudFiles").option("cloudFiles.format", "json").load(path)` |
| **File Discovery** | **Directory Listing:** Scans the full cloud storage directory repeatedly on every micro-batch. | **File Notification or Incremental Listing:** Uses cloud queues (Azure Event Grid / AWS SQS) or internal RocksDB metadata tracking to find only new files. |
| **Scalability** | Drastically degrades as directory file count grows into millions of files (high cloud API latency). | Scales to billions of files without performance degradation. |
| **Schema Management** | Strict requirement: must supply an explicit schema; fails on schema mismatches. | **Schema Inference & Evolution:** Automatically infers types, adapts to newly added columns, and diverts malformed data to `_rescued_data`. |
| **Ecosystem** | Open-source Apache Spark (any cloud, on-prem, Hadoop). | Proprietary to the Databricks Runtime. |
| **Kafka Support** | Native (`format("kafka")`). | **Not supported.** Auto Loader cannot read streaming message brokers; it only reads storage files (ADLS, S3, GCS). |

---

#### 3. Side-by-Side Code Examples

##### Standard Structured Streaming (File Source)

```python
# Requires rigid schema, scans all files repeatedly
df_spark = (
    spark.readStream
    .schema(my_strict_schema)
    .format("json")
    .load("/mnt/lake/raw_transactions/")
)

```

##### Auto Loader

```python
# Auto-infers schema, evolves dynamically, tracks file queue efficiently
df_autoloader = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/mnt/lake/schemas/transactions")
    .load("/mnt/lake/raw_transactions/")
)

```

---

#### 4. The 30-Second Interview Answer

- Auto Loader is not a separate engine; it is a Databricks-optimized data source connector built on top of Spark Structured Streaming. While standard Structured Streaming relies on expensive full-directory scans and rigid schemas to read files, Auto Loader uses cloud file notification services (like Azure Event Grid) along with automatic schema inference and evolution, making large-scale cloud file ingestion significantly faster, cheaper, and more reliable.

### Why is Auto Loader needed for cloud files but not for Kafka or Delta Lake?

Structured Streaming's performance issue was **never with streaming engines like Kafka**; the bottleneck was specifically with **cloud file systems** (ADLS, S3, GCS).

Auto Loader was created to fix file storage limitations, while Structured Streaming handles Kafka and Delta Lake natively with high performance.

---

#### 1. Structured Streaming with Kafka: High Throughput & Native Speed

Kafka does not store data as static files in folders; it is an append-only distributed log in memory and disk partitions.

* **No File Listing Overhead:** Structured Streaming does not need to scan directories. It simply connects to Kafka brokers via TCP sockets and asks: *"Give me messages from partition 0 starting at offset 4500."*
* **Direct Partition Mapping:** Spark maps Kafka partitions directly to Spark executor tasks (e.g., a Kafka topic with 12 partitions maps 1:1 to 12 parallel Spark tasks).
* **Throughput & Latency:** Structured Streaming can easily ingest hundreds of thousands to millions of records per second from Kafka with end-to-end latency as low as 100 milliseconds to a few seconds in micro-batch mode.
* **Why Auto Loader is Irrelevant Here:** Auto Loader has no concept of offsets, consumer groups, or socket streams. It only knows how to track files in cloud buckets.

---

#### 2. Structured Streaming with Delta Lake (Delta as a Source): Native & Fast

When using Delta Lake as a streaming source (`spark.readStream.format("delta").load(...)`):

* **Delta Transaction Log (`_delta_log`):** Spark does not crawl storage folders to find new Parquet files. It reads the JSON/checkpoint commits inside the Delta log to identify exactly which new data files were added in the latest commit.
* **Performance:** Because it reads a lightweight transaction log rather than scanning millions of file paths on ADLS/S3, Structured Streaming on Delta tables is fast and scalable without Auto Loader.

---

#### 3. Why Structured Streaming Needed Auto Loader ONLY for Raw Files

The reason Databricks invented Auto Loader was the **cloud storage API bottleneck**:

* Standard Structured Streaming reading raw files (`format("json")`, `format("parquet")`) must call the cloud storage API (e.g., Azure Blob/ADLS API) to run `LIST` operations on the directory.
* When a bucket accumulates 500,000+ files, running `LIST /landing/` on every micro-batch takes several minutes just to discover 10 new files.
* **Auto Loader's Sole Mission:** Replace expensive `LIST` API directory calls with **Azure Event Grid / AWS SQS notifications** or an internal RocksDB index, while dynamically handling dirty schemas (`_rescued_data`).

---

#### Performance Summary Across Sources

| Source Type | Source Connector | Performance Characteristics | Is Auto Loader Needed? |
| --- | --- | --- | --- |
| **Apache Kafka** | `format("kafka")` | Extremely fast. High throughput, low latency via offset pull over network. | **No** (Auto Loader cannot read Kafka). |
| **Delta Lake Table** | `format("delta")` | Extremely fast. Directly reads atomic transaction log files. | **No** (Delta already has its own log index). |
| **Raw Cloud Files** (JSON, CSV, Parquet in ADLS) | `format("parquet")` / `format("json")` | **Slow & degrades** as file counts grow (directory listing bottleneck). | **Yes -> Use Auto Loader (`format("cloudFiles")`)** to bypass listings. |
| **Message Queues** (Azure Event Hubs) | `format("eventhubs")` | Extremely fast. Similar partition-to-task mapping as Kafka. | **No.** |

---

#### Core Takeaway

Auto Loader is an optimization **only for file-based ingestion from object storage**. When connecting to real streaming backbones like **Kafka** or querying **Delta Lake** tables, native **Structured Streaming** operates at peak performance on its own.